# Phase - 5: Clustering into Priority Tiers (England_final_light)

**Goal:** Group England LSOAs into actionable priority tiers based on structural demand (no severity_weighted_count).

**Clustering features:** `risk_score_scaled`, `employment_deprivation`, `ntl_mean_radiance`, `resolution_rate`

**Outputs saved to:** `england_final_light/outputs/phase5/`
- `phase5_clusters.parquet`: LSOA-level cluster assignments
- `phase5_cluster_profiles.csv`: summary statistics per cluster
- `5a_silhouette.png`, `5b_gmm_criteria.png`, `5c_cluster_profiles.png`, `5c_map.png`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from collections import Counter

from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.preprocessing import MinMaxScaler
import geopandas as gpd

print('Imports OK')

In [ ]:
# Dataset Loading
BASE = Path('Dataset path')
P4   = BASE / 'england_final_light' / 'outputs' / 'phase4'
OUT  = BASE / 'england_final_light' / 'outputs' / 'phase5'
OUT.mkdir(parents=True, exist_ok=True)

SHP_FILE = BASE / 'data' / 'Lower_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V5_-7203918579177758597' / 'LSOA_2021_EW_BGC_V5.shp'

# Cluster colours (Tier 1 = highest structural demand)
TIER_COLOURS = {
    1: '#D62728',
    2: '#FF7F0E',
    3: '#BCBD22',
    4: '#2CA02C',
    5: '#1F77B4',
    6: '#9467BD',
}

print('Output folder:', OUT)

## Section-1: Load Data

In [ ]:
risk = pd.read_parquet(P4 / 'phase4_risk_scores.parquet')
print(f'Loaded: {risk.shape}')
print(f'Columns: {risk.columns.tolist()}')
print(f'Nulls: {risk.isnull().sum().sum()}')
risk.head(3)

In [ ]:
# Clustering features: risk score + structural features (no severity_weighted_count)
CLUSTER_FEATURES = [
    'risk_score_scaled',
    'employment_deprivation',
    'resolution_rate',
    'ntl_mean_radiance',
]

scaler = MinMaxScaler()
X = pd.DataFrame(
    scaler.fit_transform(risk[CLUSTER_FEATURES]),
    columns=CLUSTER_FEATURES,
    index=risk.index
)

print('Clustering matrix shape:', X.shape)
print('Feature ranges (should all be 0-1):')
print(X.agg(['min','max']).round(3))


## 5a. Optimal k Selection: K-Means + Silhouette

In [ ]:
print('Testing K-Means for k=2 to 6...')
k_range     = range(2, 7)
inertias    = []
silhouettes = []
km_models   = {}

# Silhouette is O(n²) memory — sample to avoid MemoryError on 33k LSOAs
SIL_SAMPLE = 5000
rng = np.random.default_rng(42)
sil_idx = rng.choice(len(X), size=min(SIL_SAMPLE, len(X)), replace=False)
X_sil = X.iloc[sil_idx]

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
    labels = km.fit_predict(X)
    inertia = km.inertia_
    sil = silhouette_score(X_sil, labels[sil_idx])
    inertias.append(inertia)
    silhouettes.append(sil)
    km_models[k] = (km, labels)
    print(f'  k={k}  inertia={inertia:.1f}  silhouette={sil:.4f}  (sil on {len(sil_idx):,} sample)')

best_k_sil = k_range[np.argmax(silhouettes)]
print(f'\nBest k by silhouette: {best_k_sil}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(list(k_range), inertias, 'o-', color='#378ADD', linewidth=2)
axes[0].set_xlabel('Number of clusters (k)', fontsize=11)
axes[0].set_ylabel('Inertia (within-cluster sum of squares)', fontsize=11)
axes[0].set_title('Elbow method — england_final_light', fontsize=12)
axes[0].spines[['top','right']].set_visible(False)

colours = ['#D62728' if k == best_k_sil else '#378ADD' for k in k_range]
bars = axes[1].bar(list(k_range), silhouettes, color=colours, edgecolor='white')
axes[1].set_xlabel('Number of clusters (k)', fontsize=11)
axes[1].set_ylabel('Silhouette score', fontsize=11)
axes[1].set_title('Silhouette score by k (red = best)', fontsize=12)
axes[1].spines[['top','right']].set_visible(False)
for bar, sil in zip(bars, silhouettes):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{sil:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('K-Means cluster selection — england_final_light', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUT / '5a_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 5a_silhouette.png')


## 5b. GMM Comparison: AIC/BIC

In [ ]:
print('Fitting Gaussian Mixture Models for k=2 to 6...')
aics = []
bics = []
gmm_models = {}

for k in k_range:
    gmm = GaussianMixture(n_components=k, random_state=42, n_init=5, max_iter=300)
    gmm.fit(X)
    aic = gmm.aic(X)
    bic = gmm.bic(X)
    aics.append(aic)
    bics.append(bic)
    gmm_models[k] = gmm
    print(f'  k={k}  AIC={aic:.1f}  BIC={bic:.1f}')

best_k_aic = k_range[np.argmin(aics)]
best_k_bic = k_range[np.argmin(bics)]
print(f'\nBest k by AIC: {best_k_aic}')
print(f'Best k by BIC: {best_k_bic}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(k_range), aics, 'o-', color='#378ADD', linewidth=2, label='AIC')
ax.plot(list(k_range), bics, 's--', color='#D85A30', linewidth=2, label='BIC')
ax.axvline(best_k_bic, color='#D85A30', alpha=0.3, linestyle=':', label=f'BIC min (k={best_k_bic})')
ax.axvline(best_k_aic, color='#378ADD', alpha=0.3, linestyle=':', label=f'AIC min (k={best_k_aic})')
ax.set_xlabel('Number of clusters (k)', fontsize=11)
ax.set_ylabel('Information criterion', fontsize=11)
ax.set_title('Gaussian Mixture Model — AIC & BIC — england_final_light', fontsize=12)
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(OUT / '5b_gmm_criteria.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 5b_gmm_criteria.png')

In [ ]:
votes = [best_k_sil, best_k_aic, best_k_bic]
vote_counts = Counter(votes)
FINAL_K = vote_counts.most_common(1)[0][0]

if vote_counts.most_common(1)[0][1] == 1:
    FINAL_K = best_k_sil

print(f'Silhouette best k: {best_k_sil}')
print(f'AIC best k:        {best_k_aic}')
print(f'BIC best k:        {best_k_bic}')
print(f'\n>>> FINAL k = {FINAL_K} <<<')

final_km, final_labels = km_models[FINAL_K]
risk['cluster_raw'] = final_labels

In [ ]:
cluster_mean_risk = (
    risk.groupby('cluster_raw')['risk_score_scaled']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
cluster_mean_risk['tier'] = range(1, FINAL_K + 1)
tier_map = dict(zip(cluster_mean_risk['cluster_raw'], cluster_mean_risk['tier']))
risk['tier'] = risk['cluster_raw'].map(tier_map)

print('Cluster → Tier mapping (by mean structural demand score):')
print(cluster_mean_risk)
print()
print('LSOAs per tier:')
print(risk['tier'].value_counts().sort_index())


## 5c. Cluster Profiling

In [ ]:
PROFILE_COLS = [
    'risk_score_scaled', 'crime_count',
    'employment_deprivation', 'resolution_rate', 'ntl_mean_radiance',
]

profile = risk.groupby('tier')[PROFILE_COLS].agg(['mean','median','std']).round(2)
print('--- Cluster Profiles ---')
print(profile.to_string())

profile_flat = risk.groupby('tier')[PROFILE_COLS].mean().round(2)
profile_flat['n_lsoas']  = risk.groupby('tier').size()
profile_flat['pct_total'] = (profile_flat['n_lsoas'] / len(risk) * 100).round(1)
profile_flat = profile_flat.reset_index()
print('\n--- Flat Profile (mean per tier) ---')
print(profile_flat.to_string(index=False))

In [ ]:
print('--- Top Local Authority Districts per Tier ---')
for tier in sorted(risk['tier'].unique()):
    top_lads = (
        risk[risk['tier'] == tier]
        .groupby('lad22nm')['risk_score_scaled']
        .mean()
        .sort_values(ascending=False)
        .head(5)
    )
    print(f'\nTier {tier}:')
    for lad, score in top_lads.items():
        n = len(risk[(risk['tier'] == tier) & (risk['lad22nm'] == lad)])
        print(f'  {lad:<30} mean_score={score:.1f}  n_lsoas={n}')

In [ ]:
plot_cols    = ['risk_score_scaled', 'employment_deprivation', 'resolution_rate', 'ntl_mean_radiance']
short_labels = ['Structural\ndemand', 'Employment\nDeprivation', 'Resolution', 'Night\nlights']

profile_norm = profile_flat.set_index('tier')[plot_cols].copy()
profile_norm = (profile_norm - profile_norm.min()) / (profile_norm.max() - profile_norm.min())

_fmt = lambda v: (f'{float(v):,.0f}' if abs(float(v))>=1000 else f'{float(v):.0f}' if abs(float(v))>=10 else f'{float(v):.1f}')
fig, axes = plt.subplots(1, FINAL_K, figsize=(4 * FINAL_K, 5), sharey=True)
if FINAL_K == 1:
    axes = [axes]

for tier, ax in zip(sorted(profile_norm.index), axes):
    colour = TIER_COLOURS.get(tier, '#888888')
    vals = profile_norm.loc[tier].values
    ax.bar(short_labels, vals, color=colour, edgecolor='white', alpha=0.85)
    _raw = profile_flat.set_index('tier').loc[tier, plot_cols].values
    for _xi, (_nv, _rv) in enumerate(zip(vals, _raw)):
        ax.text(_xi, _nv + 0.02, _fmt(_rv), ha='center', va='bottom', fontsize=6, rotation=90, clip_on=False)
    n   = int(profile_flat.loc[profile_flat['tier'] == tier, 'n_lsoas'].values[0])
    pct = profile_flat.loc[profile_flat['tier'] == tier, 'pct_total'].values[0]
    ax.set_title(f'Tier {tier}\n(n={n}, {pct}%)', fontsize=10, fontweight='bold')
    ax.set_ylim(0, 1.4)
    ax.tick_params(axis='x', labelsize=8)
    ax.spines[['top','right']].set_visible(False)
    if tier == 1:
        ax.set_ylabel('Normalised mean (0=low, 1=high)', fontsize=9)

plt.suptitle('Cluster profiles — england_final_light (bars normalised across tiers; labels = raw mean)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(OUT / '5c_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 5c_cluster_profiles.png')

In [ ]:
# Silhouette samples on same subsample used during k selection
sil_vals_sample = silhouette_samples(X_sil, final_labels[sil_idx])
overall_sil = sil_vals_sample.mean()

# Store per-LSOA silhouette as NaN for non-sampled rows, actual value for sampled rows
full_sil = np.full(len(X), np.nan)
full_sil[sil_idx] = sil_vals_sample
risk['silhouette'] = full_sil

fig, ax = plt.subplots(figsize=(8, 5))
y_lower = 10
for tier in sorted(risk['tier'].unique()):
    colour   = TIER_COLOURS.get(tier, '#888888')
    # Use only sampled rows for the plot
    mask     = (risk['tier'] == tier) & (~np.isnan(risk['silhouette']))
    tier_sil = np.sort(risk.loc[mask, 'silhouette'].values)
    size     = tier_sil.shape[0]
    if size == 0:
        continue
    y_upper  = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, tier_sil,
                     facecolor=colour, edgecolor=colour, alpha=0.7)
    ax.text(-0.05, y_lower + size/2, f'T{tier}', ha='right', va='center', fontsize=9)
    y_lower = y_upper + 10

ax.axvline(overall_sil, color='red', linestyle='--', linewidth=1.5,
           label=f'Mean silhouette = {overall_sil:.3f} (n={len(sil_idx):,} sample)')
ax.set_xlabel('Silhouette coefficient', fontsize=11)
ax.set_ylabel('LSOA (grouped by tier, sampled)', fontsize=11)
ax.set_title(f'Silhouette plot — k={FINAL_K} clusters — england_final_light', fontsize=12)
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(OUT / '5a_silhouette_detail.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 5a_silhouette_detail.png')

## Section-2: Choropleth Map

In [ ]:
if not SHP_FILE.exists():
    raise FileNotFoundError(
        f'ONS LSOA boundary shapefile not found at:\n  {SHP_FILE}'
    )

print('Loading ONS LSOA boundaries...')
gdf = gpd.read_file(SHP_FILE)
gdf = gdf[gdf['LSOA21CD'].str.startswith('E')].copy()
gdf = gdf[['LSOA21CD', 'geometry']].rename(columns={'LSOA21CD': 'lsoa21cd'})
gdf = gdf.reset_index(drop=True)
print(f'England LSOA boundaries: {gdf.shape}  CRS: {gdf.crs}')

In [ ]:
gdf_merged = gdf.merge(
    risk[['lsoa21cd', 'tier', 'risk_score_scaled']],
    on='lsoa21cd',
    how='left'
)
print(f'Merged GDF shape: {gdf_merged.shape}')
print(f'Unmatched LSOAs: {gdf_merged["tier"].isnull().sum()}')

gdf_merged = gdf_merged.to_crs(epsg=4326)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 16))

tiers_sorted = sorted(risk['tier'].unique())
for tier in tiers_sorted:
    colour = TIER_COLOURS.get(int(tier), '#888888')
    gdf_merged[gdf_merged['tier'] == tier].plot(
        ax=ax, color=colour, linewidth=0.02, edgecolor='white', alpha=0.85
    )

unmatched = gdf_merged[gdf_merged['tier'].isnull()]
if len(unmatched) > 0:
    unmatched.plot(ax=ax, color='#cccccc', linewidth=0.02, edgecolor='white')

tier_labels = {
    1: 'Tier 1 — Critical structural demand',
    2: 'Tier 2 — Very high structural demand',
    3: 'Tier 3 — High structural demand',
    4: 'Tier 4 — Moderate structural demand',
    5: 'Tier 5 — Low structural demand',
    6: 'Tier 6 — Minimal structural demand',
}
patches = [
    mpatches.Patch(
        color=TIER_COLOURS.get(int(t), '#888888'),
        label=tier_labels.get(int(t), f'Tier {int(t)}')
    )
    for t in tiers_sorted
]
ax.legend(handles=patches, loc='lower left', fontsize=10, framealpha=0.9)

ax.set_title('England LSOA Structural Demand Tiers (no severity weighting)', fontsize=16, fontweight='bold', pad=15)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUT / '5c_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 5c_map.png')


## Section-3: Save Outputs

In [ ]:
cluster_out = risk[['lsoa21cd','lsoa21nm','lad22nm','tier',
                     'risk_score_scaled','crime_count',
                     'employment_deprivation','ntl_mean_radiance',
                     'resolution_rate','silhouette']].copy()
cluster_out.to_parquet(OUT / 'phase5_clusters.parquet', index=False)
print(f'Saved phase5_clusters.parquet — {cluster_out.shape}')

profile_flat.to_csv(OUT / 'phase5_cluster_profiles.csv', index=False)
print(f'Saved phase5_cluster_profiles.csv — {profile_flat.shape}')


## Section-4: Alternative k=4 Solution

In [ ]:
K4 = 4
km4 = KMeans(n_clusters=K4, random_state=42, n_init=20, max_iter=500)
labels4      = km4.fit_predict(X)
sil4         = silhouette_score(X_sil, labels4[sil_idx])
sil4_samples_sub = silhouette_samples(X_sil, labels4[sil_idx])

# Store NaN for non-sampled rows
full_sil4 = np.full(len(X), np.nan)
full_sil4[sil_idx] = sil4_samples_sub

risk4 = risk[['lsoa21cd','lsoa21nm','lad22nm','crime_count',
              'employment_deprivation','ntl_mean_radiance',
              'resolution_rate','risk_score_scaled',
              'nb_predicted','nb_ratio']].copy()
risk4['cluster_raw'] = labels4
risk4['silhouette']  = full_sil4

cluster_mean4 = (
    risk4.groupby('cluster_raw')['risk_score_scaled']
    .mean().sort_values(ascending=False).reset_index()
)
cluster_mean4['tier'] = range(1, K4 + 1)
tier_map4 = dict(zip(cluster_mean4['cluster_raw'], cluster_mean4['tier']))
risk4['tier'] = risk4['cluster_raw'].map(tier_map4)

print(f'k=4  silhouette={sil4:.4f}  (sample n={len(sil_idx):,})')
print('\nLSOAs per tier:')
print(risk4['tier'].value_counts().sort_index())
print('\nMean structural demand score per tier:')
print(risk4.groupby('tier')['risk_score_scaled'].mean().round(2))

In [ ]:
TIER4_COLOURS = {1: '#D62728', 2: '#FF7F0E', 3: '#2CA02C', 4: '#1F77B4'}
TIER4_LABELS  = {
    1: 'Tier 1 — Critical structural demand',
    2: 'Tier 2 — High structural demand',
    3: 'Tier 3 — Moderate structural demand',
    4: 'Tier 4 — Low structural demand',
}

PROFILE4_COLS = ['risk_score_scaled','crime_count','employment_deprivation','resolution_rate','ntl_mean_radiance']

profile4 = risk4.groupby('tier')[PROFILE4_COLS].mean().round(2)
profile4['n_lsoas']   = risk4.groupby('tier').size()
profile4['pct_total'] = (profile4['n_lsoas'] / len(risk4) * 100).round(1)
profile4 = profile4.reset_index()

plot_cols4    = ['risk_score_scaled','employment_deprivation','resolution_rate','ntl_mean_radiance']
short_labels4 = ['Structural\ndemand','Employment\nDeprivation','Resolution','Night\nlights']

profile4_norm = profile4.set_index('tier')[plot_cols4].copy()
profile4_norm = (profile4_norm - profile4_norm.min()) / (profile4_norm.max() - profile4_norm.min())

_fmt = lambda v: (f'{float(v):,.0f}' if abs(float(v))>=1000 else f'{float(v):.0f}' if abs(float(v))>=10 else f'{float(v):.1f}')
fig, axes = plt.subplots(1, K4, figsize=(4 * K4, 5), sharey=True)
for tier, ax in zip(range(1, K4 + 1), axes):
    colour = TIER4_COLOURS[tier]
    vals   = profile4_norm.loc[tier].values
    ax.bar(short_labels4, vals, color=colour, edgecolor='white', alpha=0.85)
    _raw = profile4.set_index('tier').loc[tier, plot_cols4].values
    for _xi, (_nv, _rv) in enumerate(zip(vals, _raw)):
        ax.text(_xi, _nv + 0.02, _fmt(_rv), ha='center', va='bottom', fontsize=6, rotation=90, clip_on=False)
    n   = int(profile4.loc[profile4['tier'] == tier, 'n_lsoas'].values[0])
    pct = profile4.loc[profile4['tier'] == tier, 'pct_total'].values[0]
    ax.set_title(f'Tier {tier}\n(n={n}, {pct}%)', fontsize=10, fontweight='bold')
    ax.set_ylim(0, 1.4)
    ax.tick_params(axis='x', labelsize=8)
    ax.spines[['top','right']].set_visible(False)
    if tier == 1:
        ax.set_ylabel('Normalised mean (0=low, 1=high)', fontsize=9)

plt.suptitle('k=4 Cluster profiles — england_final_light (bars normalised across tiers; labels = raw mean)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(OUT / '5d_k4_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 5d_k4_cluster_profiles.png')

In [ ]:
gdf_k4 = gdf.merge(risk4[['lsoa21cd','tier','risk_score_scaled']], on='lsoa21cd', how='left')
gdf_k4 = gdf_k4.to_crs(epsg=4326)

fig, ax = plt.subplots(figsize=(14, 16))
for tier in range(1, K4 + 1):
    gdf_k4[gdf_k4['tier'] == tier].plot(
        ax=ax, color=TIER4_COLOURS[tier], linewidth=0.02, edgecolor='white', alpha=0.85
    )
unmatched4 = gdf_k4[gdf_k4['tier'].isnull()]
if len(unmatched4) > 0:
    unmatched4.plot(ax=ax, color='#cccccc', linewidth=0.02, edgecolor='white')

patches4 = [mpatches.Patch(color=TIER4_COLOURS[t], label=TIER4_LABELS[t]) for t in range(1, K4 + 1)]
ax.legend(handles=patches4, loc='lower left', fontsize=10, framealpha=0.9)
ax.set_title('England LSOA Structural Demand Tiers (k=4, no severity)', fontsize=16, fontweight='bold', pad=15)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUT / '5d_k4_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 5d_k4_map.png')

In [ ]:
risk4[['lsoa21cd','lsoa21nm','lad22nm','tier','risk_score_scaled','crime_count',
       'employment_deprivation','ntl_mean_radiance','resolution_rate',
       'silhouette']].to_parquet(OUT / 'phase5_clusters_k4.parquet', index=False)
print(f'Saved phase5_clusters_k4.parquet')

profile4.to_csv(OUT / 'phase5_cluster_profiles_k4.csv', index=False)
print(f'Saved phase5_cluster_profiles_k4.csv')

print(f'\nSilhouette score (k=4): {sil4:.4f}  |  (k={FINAL_K}): {overall_sil:.4f}')

In [ ]:
print('=' * 58)
print('PHASE 5 SUMMARY (england_final_light)')
print('=' * 58)
print(f'Total LSOAs:               {len(risk)}')
print(f'Final k (tiers):           {FINAL_K}')
print(f'Silhouette score (k={FINAL_K}):   {overall_sil:.4f}')
print(f'Best k — silhouette:       {best_k_sil}')
print(f'Best k — AIC:              {best_k_aic}')
print(f'Best k — BIC:              {best_k_bic}')
print()
print('LSOAs per tier:')
for _, row in profile_flat.iterrows():
    print(f"  Tier {int(row['tier'])}: {int(row['n_lsoas'])} LSOAs ({row['pct_total']}%)  mean_score={row['risk_score_scaled']:.1f}")
print('=' * 58)
print(f'Outputs saved to: {OUT}')